In [1]:
import sqlite3
import json
import os

# Paths
db_path = r"C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.DB"
json_folder = r"C:\Users\liam_\Documents\Games DE Project\Zelda ETL\Landing\landing"

conn = sqlite3.connect(db_path, timeout=30)
cursor = conn.cursor()

for filename in os.listdir(json_folder):
    if filename.endswith(".json"):
        file_path = os.path.join(json_folder, filename)

        # Table name from file
        table_name = "stg_" + filename.replace(".json", "").replace("_raw", "")

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

            # Handle API wrapper
            if isinstance(data, dict) and "data" in data:
                data = data["data"]

            entries = data if isinstance(data, list) else [data]

        # Detect columns for THIS file only
        columns = set()
        for entry in entries:
            if isinstance(entry, dict):
                columns.update(entry.keys())

        columns = list(columns)

        # Drop table if exists (fresh staging load)
        cursor.execute(f'DROP TABLE IF EXISTS "{table_name}"')

        # Create table WITHOUT primary key
        columns_def = ", ".join([f'"{col}" TEXT' for col in columns])

        cursor.execute(f'''
        CREATE TABLE "{table_name}" (
            {columns_def}
        )
        ''')

        # Insert rows
        for entry in entries:
            if isinstance(entry, dict):
                values = [str(entry.get(col, None)) for col in columns]
                placeholders = ", ".join(["?"] * len(columns))

                cursor.execute(
                    f'INSERT INTO "{table_name}" ({", ".join(columns)}) VALUES ({placeholders})',
                    values
                )

        conn.commit()
        print(f" Loaded {len(entries)} rows into {table_name}")

conn.close()


 Loaded 255 rows into stg_bosses
 Loaded 1657 rows into stg_characters
 Loaded 335 rows into stg_dungeons
 Loaded 12 rows into stg_games
 Loaded 1849 rows into stg_items
 Loaded 815 rows into stg_monsters
 Loaded 1470 rows into stg_places
 Loaded 234 rows into stg_staff
